# alpha-blended-eaf: Colab Pipeline

**Before running:**
1. Set the runtime to GPU (Runtime → Change runtime type → GPU).
2. Run Cell 1 to mount Drive.
3. Run Cell 2 to set up the environment.
4. Run Cell 3 (dry run) to validate training end-to-end.
5. Run Cell 4 (full sweep) only after the dry run passes.

Re-running any cell is safe — every step has a skip-if-done guard.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Cell 2 — Environment setup
# First run: ~20-30 min. Subsequent runs: ~2-3 min.
set -e
cd /content
if [ ! -d /content/alpha-blended-eaf/.git ]; then
    git clone https://github.com/YOUR_USERNAME/alpha-blended-eaf.git
else
    cd /content/alpha-blended-eaf && git pull --ff-only && cd /content
fi
bash /content/alpha-blended-eaf/colab_run.sh

In [ ]:
%%bash
# Cell 3 — Dry run (1 config x 1 fold x 2 epochs, ~10 min)
# Run this before the full sweep. Success = prints metrics + 'Stage 1 sweep complete.'
set -e
REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results
KFOLD=/content/src/Japan_kfold
python "${REPO}/runner/run_sweep.py" \
    --sweep-config  "${REPO}/configs/sweep_configs.yaml" \
    --kfold-dir     "${KFOLD}" \
    --results-dir   "${DRIVE}" \
    --repo-root     "${REPO}" \
    --dry-run

In [ ]:
%%bash
# Cell 4 — Full Stage 1 sweep (4 configs x 10 folds = 40 runs)
# Run ONLY after Cell 3 passes. Safe to interrupt and resume.
set -e
REPO=/content/alpha-blended-eaf
DRIVE=/content/drive/MyDrive/Dissertation_Results/Results
KFOLD=/content/src/Japan_kfold
python "${REPO}/runner/run_sweep.py" \
    --sweep-config  "${REPO}/configs/sweep_configs.yaml" \
    --kfold-dir     "${KFOLD}" \
    --results-dir   "${DRIVE}" \
    --repo-root     "${REPO}"

## After Stage 1 completes

Results land in `MyDrive/Dissertation_Results/Results/sweep_runs/`:
```
all_runs.csv                         <- every fold, every config
stage1_eps0.1_r0.3/fold_metrics.csv  <- mean +/- std for r=0.3
stage1_eps0.1_r0.4/fold_metrics.csv  <- mean +/- std for r=0.4
stage1_eps0.1_r0.5/fold_metrics.csv  <- mean +/- std for r=0.5
stage1_eps0.1_r0.6/fold_metrics.csv  <- mean +/- std for r=0.6
```

Next steps:
1. Open `all_runs.csv`, identify the r with the best mean mAP50.
2. Edit `configs/sweep_configs.yaml`, set `stage2.fixed.r` to the winning value.
3. Commit and push.
4. Build `runner/aggregate_results.py` for summary tables and per-fold plots.